goal: when a car isn't moving, every sensor should read a known value. Whatever it reads instead of its offset (avg) and its noise (std)

what sensors should read at rest:
* gyro (angural rate) x,y,z: 0 (anything else is offset)
* accelerometer x,y: abt 0 if the groups is level, z: gravity 0
* throttle: 0
* brake: 0 unless the driver is holding it
* steering: not always 0 so we can't tell offset of steering from the standstill

from ou previous analysis (02_session) we found three stationary periods:
* the start 0-60
* pit stop 980-1200 (approx)
* end ~1976 to the end

which could help to compare them to find out if we have a stable offset that we can subtract

If a car is being pushed slowly, the front wheels would have a speed that is slower than walking, a car being pushed could reach the speed of 0.5m/s (assuming it's heavy and it's hard to be pushed) so a standstill would mean that the wheel speed is slower than that, say around 0.19m/s, so 1rad/s so that any real push would be over that value. 

hence, it would make sense to take the threashold as 1rad/s from the front_angular_velocity to determine if a car is staionary or not.

In [1]:
from src.loader import load
brake_press = load("pedal_brakes_pressure")
brake_press_front_max = brake_press[brake_press["front"] ==brake_press["front"].max()]
brake_press_front_max

,t_s,_timestamp,front,rear
49544,519.588272,1725799638964815,22.671854,18.742657


In [2]:
imu_acc = load("imu_acceleration")
imu_acc_x_max = imu_acc[(imu_acc["t_s"] >= 519.58) & (imu_acc["t_s"] <= 520.4)]
imu_acc_x_max

,t_s,_timestamp,x,y,z,imu_temperature
115937,519.580767,1725799638957310,-1.29,0.31,-0.94,0
115938,519.585227,1725799638961770,-1.23,0.09,-0.92,0
115939,519.589757,1725799638966300,-1.18,0.16,-0.95,0
115940,519.594281,1725799638970824,-1.22,0.10,-0.92,0
115941,519.598662,1725799638975205,-1.05,0.07,-0.92,0
...,...,...,...,...,...,...
116115,520.377963,1725799639754506,-0.79,0.23,-0.86,0
116116,520.382333,1725799639758876,-0.64,0.10,-0.84,0
116117,520.387001,1725799639763544,-0.75,0.35,-0.68,0
116118,520.391309,1725799639767852,-0.71,0.33,-0.81,0


in case of a lockup, the angular speed can show 0 during braking until the car stops or driver lifts up the brakes.
if we take that into account, and assuming the car decelerates at 1.2g (avg of (−1.29, −1.23, −1.18, −1.22, −1.05) in the table above) max at peak brake pressure, equivalent to (1.2*9.81) ~11.772m/s^2, and given our data, the max speed reached by the car is 23m/s, then the car comes to a stop after abt 2s, we we can assume that the car is at standstill if the front angular velocity ~0 for 3 seconds

In [3]:
from src.utils import find_standstills

In [4]:
standstill_windows = find_standstills()
standstill_windows

,start_s,end_s,duration_s
0,0.000000,13.002358,13.002358
1,20.790024,54.929918,34.139894
2,161.421000,176.054854,14.633854
3,192.778036,206.897613,14.119577
4,227.119072,242.465951,15.346879
5,977.176868,1157.391011,180.214143
6,1163.760712,1196.196530,32.435818
7,1829.968137,1908.013226,78.045089
8,1920.559415,2206.962362,286.402947


### findings
* start: 0-13 and 21-55s, smth moved teh wheels between 13 and 21s the driver released the brake and and pressed the throttle a lil (according to teh graph from 02_session notebook).
* driver was holding brake ~4bar and released at ~13 before moving off
* three ~15s stops at 161, 193, and 227s -> why? blue flag?
* pit: 977-1157 and 1164-1196 split by a brief movement around 1157 and 1164s, probably being repositioned in the pit
* end: 1830-1908s then 1921-2207s: the gap there is the car being pushed


investigate why there are three ~15s stops

In [6]:
import plotly.graph_objects as go
from src.config import PIT_MIN_S
from src.loader import load, load_centerline

cl = load_centerline()
pos = load("vehicle_position")
stops = standstill_windows[standstill_windows["duration_s"] < PIT_MIN_S]

fig = go.Figure()
fig.add_trace(go.Scatter(x=cl["x"], y=cl["y"], mode="lines",
                         line=dict(color="lightgray"), name="track"))

for i, w in stops.iterrows():
    p = pos[(pos["t_s"] >= w["start_s"]) & (pos["t_s"] <= w["end_s"])]
    if p.empty:  
        continue
    fig.add_trace(go.Scatter(
        x=[p["x"].mean()], y=[p["y"].mean()], mode="markers+text",
        marker=dict(size=12), text=[f"{w['start_s']:.0f} s"],
        textposition="top center", name=f"stop at {w['start_s']:.0f} s",
    ))

fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(title="Where the short stops happened", height=700)
fig.show()

they don't happen to be in the same location on track, let's see the other signals to find a patterns

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.loader import load
from src.config import WHEEL_RADIUS

# the three on-track stops
track_stops = standstill_windows[(standstill_windows["start_s"] > 100) & (standstill_windows["start_s"] < 300)]

thr = load("pedal_throttle")
brk = load("pedal_brakes_pressure")
gyro = load("imu_angular_rate")
wheels = load("front_angular_velocity")
curv = load("vehicle_curvilinear_coordinates")

wheels["speed_ms"] = (wheels["fl"] + wheels["fr"]) / 2 * WHEEL_RADIUS

signals = [
    (thr, "throttle", "throttle"),
    (brk, "front", "front brake (bar)"),
    (gyro, "z", "yaw rate (deg/s)"),
    (wheels, "speed_ms", "speed (m/s)"),
    (curv, "n", "n, distance from centerline (m)"),
]

for _, stop in track_stops.iterrows():
    t0, t1 = stop["start_s"] - 10, stop["end_s"] + 10

    fig = make_subplots(rows=len(signals), cols=1, shared_xaxes=True,
                        subplot_titles=[label for _, _, label in signals])
    for i, (df, col, label) in enumerate(signals, start=1):
        w = df[(df["t_s"] >= t0) & (df["t_s"] <= t1)]
        fig.add_trace(go.Scatter(x=w["t_s"], y=w[col], name=label), row=i, col=1)

    # shade the stop itself
    fig.add_vrect(x0=stop["start_s"], x1=stop["end_s"],
                  fillcolor="gray", opacity=0.15, line_width=0)

    fig.update_layout(height=900, showlegend=False,
                      title=f"Stop at {stop['start_s']:.0f}–{stop['end_s']:.0f} s")
    fig.update_xaxes(title_text="t_s (s)", row=len(signals), col=1)
    fig.update_xaxes(showticklabels=True)
    fig.show()

It seems like the car stopped because the driver came to a stop (low throttle and high brake pressue), the reason why a driver would come to a stop 3 times around the track in different location that is not due to a mechanicl failure, and looking at n, diatnce from teh certline in meters, we can note that the factor in common is that the drivers pulls to the side by about 2m-6m to the left, stops for abt 15s and then accelerates again. this could very well be a **blue flag** where the driver has to pull aside to let faster card through.
of course this is just a hypothesis, we can't rule out other factors that may have happened. 

Looking for offset in sensors

In [ ]:
SENSORS = {
    "imu_angular_rate": ["x", "y", "z"],
    "imu_acceleration": ["x", "y", "z"], # imu_temp is not gonna be included bc it's all 0
    "front_angular_velocity": ["fl", "fr"],
    "inv_l_id_a8_n_actual_filt": ["n_act_filt"],
    "inv_r_id_a8_n_actual_filt": ["n_act_filt"],
    "inv_l_id_27_iq_actual": ["iq_act_filt"],
    "inv_r_id_27_iq_actual": ["iq_act_filt"],
    "pedal_throttle": ["throttle"],
    "pedal_brakes_pressure": ["front", "rear"],
    "hv_power": ["power"],
}

In [ ]:
# i want to create a table that has 
# window | file | column | mean | std | n 
# where mean is the mean of the column in that window
# the std is the standard deviation of that column in that window
# and n is the sample count in that window
# i will create it in profiler.py then use it here
# here it is
from src.utils import standstill_stats

In [ ]:
stats = standstill_stats()

In [ ]:
stats

,window,start_s,file,column,mean,std,n
0,0,0.000000,imu_angular_rate,x,1.160781,0.084023,2881
1,0,0.000000,imu_angular_rate,y,-0.250902,0.098941,2881
2,0,0.000000,imu_angular_rate,z,1.023888,0.068993,2881
3,1,20.790024,imu_angular_rate,x,1.149519,0.090633,7634
4,1,20.790024,imu_angular_rate,y,-0.247760,0.133117,7634
...,...,...,...,...,...,...,...
139,4,227.119072,hv_power,power,0.098564,0.080333,266
140,5,977.176868,hv_power,power,0.009801,0.052140,2766
141,6,1163.760712,hv_power,power,0.202063,0.023639,550
142,7,1829.968137,hv_power,power,-0.001333,0.000743,1217


In [ ]:
# compare across windows
# pivot to see things clearly
stats.pivot_table(index=["file", "column"], columns="window", values="mean")

window                                        0         1         2         3  \
file                      column                                                
front_angular_velocity    fl           0.004517  0.003135  0.003956  0.006108   
                          fr           0.003689  0.002297  0.003543  0.006460   
hv_power                  power        0.137607  0.209457 -0.008984  0.120708   
imu_acceleration          x           -0.063543 -0.064490 -0.076894 -0.078584   
                          y           -0.020944 -0.020206 -0.038609 -0.116654   
                          z           -0.969594 -0.969471 -0.966940 -0.959943   
imu_angular_rate          x            1.160781  1.149519  1.130083  1.124208   
                          y           -0.250902 -0.247760 -0.245634 -0.263622   
                          z            1.023888  1.018931  1.027420  1.022278   
inv_l_id_27_iq_actual     iq_act_filt  0.166404  0.053425  0.099112  0.088237   
inv_l_id_a8_n_actual_filt n_act_filt   0.009938  0.009249  0.015013  0.016944   
inv_r_id_27_iq_actual     iq_act_filt  0.167467  0.050205  0.106269  0.096398   
inv_r_id_a8_n_actual_filt n_act_filt   0.008876  0.007462  0.014322  0.015233   
pedal_brakes_pressure     front        3.556918  2.371987  5.195430  3.169134   
                          rear         2.080581  1.058474  3.416432  1.961850   
pedal_throttle            throttle     0.003395  0.000931  0.001742  0.001695   

window                                        4         5         6         7  \
file                      column                                                
front_angular_velocity    fl           0.006259 -0.003780 -0.007801  0.002211   
                          fr           0.006052 -0.003756 -0.006718  0.001311   
hv_power                  power        0.098564  0.009801  0.202063 -0.001333   
imu_acceleration          x           -0.086316 -0.077303 -0.062228 -0.072627   
                          y           -0.097464 -0.032778 -0.020622 -0.007133   
                          z           -0.961900 -0.969828 -0.970063 -0.971069   
imu_angular_rate          x            1.114392  1.042311  1.031223  0.979705   
                          y           -0.276730 -0.290123 -0.285359 -0.315548   
                          z            1.011504  1.008271  1.012904  0.994137   
inv_l_id_27_iq_actual     iq_act_filt  0.072790  0.008180  0.045159  0.006030   
inv_l_id_a8_n_actual_filt n_act_filt   0.015407 -0.000214 -0.000126  0.004413   
inv_r_id_27_iq_actual     iq_act_filt  0.066939  0.015448  0.050641 -0.000470   
inv_r_id_a8_n_actual_filt n_act_filt   0.015115 -0.000292 -0.000017  0.003535   
pedal_brakes_pressure     front        3.105681  1.312924  0.073984  0.881079   
                          rear         1.854728  0.983740  0.029772  0.305818   
pedal_throttle            throttle     0.001360  0.000185  0.000763  0.000000   

window                                        8  
file                      column                 
front_angular_velocity    fl          -0.002550  
                          fr          -0.002613  
hv_power                  power       -0.001325  
imu_acceleration          x           -0.068249  
                          y           -0.017882  
                          z           -0.970457  
imu_angular_rate          x            0.953836  
                          y           -0.309421  
                          z            0.983774  
inv_l_id_27_iq_actual     iq_act_filt  0.001607  
inv_l_id_a8_n_actual_filt n_act_filt   0.000634  
inv_r_id_27_iq_actual     iq_act_filt -0.002140  
inv_r_id_a8_n_actual_filt n_act_filt   0.000604  
pedal_brakes_pressure     front        0.049420  
                          rear         0.015623  
pedal_throttle            throttle     0.000000

## Analysis
* so for the decisions that were made for standstill: front wheel speed < 1 rad/s for >= 3s
* we are excluding the steering, and the estimator outputs bc they are computed not measured, and the GPS as well.
Offset analysis:
* brake: rest value is ~0.05bar vs 15-25 during peak braking -> the rest value is negligible.
* power: 0.1-0.2kW when the car active which is considered to be a real consumption so no need to fix anything.
* motor current: ~0.1 A vs 63 A max -> negligible 
* wheel speed (front_angular_velocity and inverter data): ~0.01 rad/s -> negligible
* throttle: <= 0.003 vs 0.95, 0 when driver is out of the car at the end -> negligible
* IMU_acc_x: -0.07 g in every window, all locations → mounting angle / bias, ~6% of peak braking -> remove it
* IMU_acc_y: ~-0.02g but windows 3–4 ~-0.1g so assumption is the car on cambered track so we exclude them from the analysis. -> neglect it bc ~0.02g against lateral peaks of ~2g is about 1%, compare that with x at 0.07 g, about 6% of braking, which is why x gets subtracted and y doesn't. Same rule, different sizes.
* IMU_acc_z: ~-0.97 g vs expected -1 g -> it's fine we can neglect it since vertical acc is not used for braking or cornering analysis while we care about the x.
* gyro: offsets x ~ 1.0, y ~ -0.28, z ~ 1.0 deg/s, slow drift (temperature drift) -> options: 
    - we compute the average of the means and we subtract that obtained value (we will go with this one for simplicity)
    - per window: subtract the 1st standstill mean from that window + the first moving period, subtract the second standstill mean from the second standstill and from the second moving period, and so on until the end. OR the other around.
    - interpolate the two consecutive standstill means and use that as the offset for the moving part in between.
